# TF-IDF + Logistic Regression Model

Baseline sentiment classification using TF-IDF vectorization and Logistic Regression.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

## Text Cleaning

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

## Load Data

In [ ]:
train_df = pd.read_csv('data/train.csv').dropna(subset=['review'])
val_df = pd.read_csv('data/val.csv').dropna(subset=['review'])

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")

## Prepare Features and Labels

In [ ]:
X_train = train_df['review'].apply(clean_text).values
y_train = (train_df['sentiment'] == 'positive').astype(int).values

X_val = val_df['review'].apply(clean_text).values
y_val = (val_df['sentiment'] == 'positive').astype(int).values

## Vectorize Text with TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)

print(f"Feature matrix shape: {X_train_vec.shape}")

## Train Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_vec, y_train)
print("Training complete!")

## Evaluate on Validation Set

In [ ]:
val_preds = lr_model.predict(X_val_vec)
print(f"Validation Accuracy: {accuracy_score(y_val, val_preds):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, val_preds, target_names=['Negative (0)', 'Positive (1)']))

## Save Model Artifacts

In [ ]:
model_dir = 'models/tfidf_lr'
os.makedirs(model_dir, exist_ok=True)

with open(os.path.join(model_dir, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(vectorizer, f)
with open(os.path.join(model_dir, 'lr_model.pkl'), 'wb') as f:
    pickle.dump(lr_model, f)

print("Model saved successfully.")

## Quick Prediction Test

In [ ]:
def predict_sentiment(text):
    clean = clean_text(text)
    vec = vectorizer.transform([clean])
    pred = lr_model.predict(vec)[0]
    prob = lr_model.predict_proba(vec)[0][pred]
    sentiment = "positive" if pred == 1 else "negative"
    return sentiment, prob

# Test
text = "This movie was absolutely wonderful! Great acting and beautiful cinematography."
sentiment, confidence = predict_sentiment(text)
print(f"Text: {text}")
print(f"Prediction: {sentiment} (confidence: {confidence:.4f})")